<a href="https://colab.research.google.com/github/knyazevaanna16-arch/programming_tsib_251_12/blob/main/lab_04_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# task_04_02_12

# Домен / Сфера: Клиника
# Дочерние классы 1 (Наследуют БК 1): Consultation (врач), LabTest (тип анализа)
# Дочерний класс 2 (Наследует БК 2): InsuredPatient (страх. полис)
# Контейнер: Visit
# Полиморфный метод (Бизнес-логика): calculate_final_bill()

# Выполнил: Князева А.П.
# Группа ЦИБ 251

# ==============================================================================
# БАЗОВЫЕ КЛАССЫ (из ЛР 4.1, с доработками для ЛР 4.2)
# ==============================================================================

class MedicalService:
    """Базовый класс 1: Медицинская услуга (Сущность)."""

    def __init__(self, name, cost):
        self.name = name
        self.cost = cost

    def calculate_final_bill(self):
        """
        Полиморфный метод.
        В базовом классе возвращает полную стоимость.
        Переопределяется в дочерних классах.
        """
        return self.cost

    def __str__(self):
        return f"Услуга: {self.name:<20} | Базовая цена: {self.cost} руб."


class Patient:
    """Базовый класс 2: Пациент (Субъект)."""

    def __init__(self, full_name, card_number):
        self.full_name = full_name
        self.card_number = card_number

    def __str__(self):
        return f"Пациент: {self.full_name:<25} | Карта №: {self.card_number}"


# ==============================================================================
# ЭТАП 1 & 2: НАСЛЕДОВАНИЕ И ПОЛИМОРФИЗМ (Дочерние классы 1)
# ==============================================================================

class Consultation(MedicalService):
    """
    Дочерний класс 1: Консультация врача.
    Наследует MedicalService.
    Специфичное поле: doctor_name (имя врача).
    """

    def __init__(self, name, cost, doctor_name):
        super().__init__(name, cost)
        self.doctor_name = doctor_name

    def calculate_final_bill(self):
        """
        Переопределение бизнес-логики.
        Для консультаций применяется наценка 10% за квалификацию врача.
        """
        return self.cost * 1.10

    def __str__(self):
        base_str = super().__str__()
        return f"{base_str} | Врач: {self.doctor_name}"


class LabTest(MedicalService):
    """
    Дочерний класс 1: Лабораторный анализ.
    Наследует MedicalService.
    Специфичное поле: test_type (тип анализа).
    """

    def __init__(self, name, cost, test_type):
        super().__init__(name, cost)
        self.test_type = test_type

    def calculate_final_bill(self):
        """
        Переопределение бизнес-логики.
        К стоимости анализа добавляется фиксированная цена расходных материалов (300 руб).
        """
        materials_cost = 300
        return self.cost + materials_cost

    def __str__(self):
        base_str = super().__str__()
        return f"{base_str} | Тип: {self.test_type}"


# ==============================================================================
# ЭТАП 1: НАСЛЕДОВАНИЕ (Дочерний класс 2)
# ==============================================================================

class InsuredPatient(Patient):
    """
    Дочерний класс 2: Застрахованный пациент.
    Наследует Patient.
    Специфичное поле: policy_number (номер полиса), discount_rate (процент скидки).
    """

    def __init__(self, full_name, card_number, policy_number, discount_rate=0.20):
        super().__init__(full_name, card_number)
        self.policy_number = policy_number
        self.discount_rate = discount_rate

    def __str__(self):
        base_str = super().__str__()
        return f"{base_str} | Полис: {self.policy_number} (Скидка: {int(self.discount_rate * 100)}%)"


# ==============================================================================
# ЭТАП 3 & 4: КОМПОЗИЦИЯ (Класс-контейнер)
# ==============================================================================

class Visit:
    """
    Класс-контейнер: Визит пациента.
    Содержит список услуг и ссылку на пациента.
    """

    def __init__(self, patient: Patient):
        self.patient = patient
        self.services = []

    def add_item(self, item: MedicalService):
        """Добавляет услугу в список визита с проверкой типа."""
        if isinstance(item, MedicalService):
            self.services.append(item)
        else:
            # Это исключение будет перехвачено блоком except TypeError ниже
            raise TypeError("Можно добавлять только объекты типа MedicalService")

    def get_detailed_report(self):
        """Формирует текстовый отчет по всем услугам с учетом полиморфного расчета."""
        report_lines = []
        report_lines.append(f"\n{'='*55}")
        report_lines.append(f"{'ОТЧЕТ ПО ВИЗИТУ':^55}")
        report_lines.append(f"{'='*55}")
        report_lines.append(f"Пациент: {self.patient.full_name}")

        if isinstance(self.patient, InsuredPatient):
            report_lines.append(f"Страховой полис: {self.patient.policy_number}")

        report_lines.append(f"{'-'*55}")
        report_lines.append(f"{'Наименование услуги':<35} | {'Итоговая цена':>15}")
        report_lines.append(f"{'-'*55}")

        for service in self.services:
            final_price = service.calculate_final_bill()
            report_lines.append(f"{service.name:<35} | {final_price:>15.2f} руб.")

        return "\n".join(report_lines)

    def calculate_total(self):
        """
        ЭТАП 4: Итоговая логика.
        1. Перебирает список услуг.
        2. Вызывает полиморфный метод calculate_final_bill() у каждой услуги.
        3. Применяет логику Дочернего класса 2 (скидка для InsuredPatient).
        """
        total_sum = 0

        for service in self.services:
            total_sum += service.calculate_final_bill()

        final_bill = total_sum

        if isinstance(self.patient, InsuredPatient):
            discount_amount = total_sum * self.patient.discount_rate
            final_bill -= discount_amount

        return final_bill


# ==============================================================================
# ЭТАП 5: ДЕМОНСТРАЦИЯ СЦЕНАРИЯ (Main) с обработкой исключений
# ==============================================================================

if __name__ == "__main__":
    try:
        regular_patient = Patient("Иванова Мария Петровна", "123-456")
        insured_patient = InsuredPatient("Смирнов Алексей Игоревич", "789-012", "POL-998877", discount_rate=0.25)

        print(regular_patient)
        print(insured_patient)

        serv_1 = Consultation("Прием терапевта", 2800, "Др. Хаус")
        serv_2 = LabTest("Анализ крови (общий)", 1200, "Гематология")
        serv_3 = Consultation("УЗИ сердца", 3500, "Др. Стрэндж")
        serv_4 = LabTest("Биохимия печени", 2200, "Биохимия")

        print("\n--- Созданные услуги ---")
        print(serv_1)
        print(serv_2)
        print(serv_3)
        print(serv_4)

        # Сценарий 1: Обычный пациент
        print("\n\n>>> СЦЕНАРИЙ 1: Обычный пациент")
        visit_1 = Visit(regular_patient)
        visit_1.add_item(serv_1)
        visit_1.add_item(serv_2)

        print(visit_1.get_detailed_report())
        total_1 = visit_1.calculate_total()
        print(f"{'-'*55}")
        print(f"{'ИТОГО К ОПЛАТЕ:':<35} | {total_1:>15.2f} руб.")
        print(f"{'='*55}")

        # Сценарий 2: Застрахованный пациент
        print("\n\n>>> СЦЕНАРИЙ 2: Застрахованный пациент (со скидкой)")
        visit_2 = Visit(insured_patient)
        visit_2.add_item(serv_1)
        visit_2.add_item(serv_2)
        visit_2.add_item(serv_3)
        visit_2.add_item(serv_4)

        print(visit_2.get_detailed_report())
        total_2 = visit_2.calculate_total()
        print(f"{'-'*55}")
        print(f"{'ИТОГО К ОПЛАТЕ (со скидкой 25%):':<35} | {total_2:>15.2f} руб.")
        print(f"{'='*55}")

        # --- ДЕМО-ТЕСТ ОБРАБОТКИ ИСКЛЮЧЕНИЙ (можно раскомментировать для защиты) ---
        # print("\n[ТЕСТ] Попытка добавить некорректный объект (строку вместо услуги):")
        # visit_1.add_item("Это не медицинская услуга")

    except TypeError as te:
        # Перехватывает ошибки типов, включая ту, что мы сами вызываем в add_item()
        print(f"Ошибка: Нарушение типизации данных: {te}")
    except ValueError as ve:
        # Перехватывает логические ошибки значений (например, если бы мы передали cost = -100)
        print(f"Ошибка: Некорректное значение атрибута: {ve}")
    except Exception as e:
        # Универсальный "страховочный" перехватчик для любых других непредвиденных ошибок
        print(f"Ошибка: Произошло исключение: {e}")

Пациент: Иванова Мария Петровна    | Карта №: 123-456
Пациент: Смирнов Алексей Игоревич  | Карта №: 789-012 | Полис: POL-998877 (Скидка: 25%)

--- Созданные услуги ---
Услуга: Прием терапевта      | Базовая цена: 2800 руб. | Врач: Др. Хаус
Услуга: Анализ крови (общий) | Базовая цена: 1200 руб. | Тип: Гематология
Услуга: УЗИ сердца           | Базовая цена: 3500 руб. | Врач: Др. Стрэндж
Услуга: Биохимия печени      | Базовая цена: 2200 руб. | Тип: Биохимия


>>> СЦЕНАРИЙ 1: Обычный пациент

                    ОТЧЕТ ПО ВИЗИТУ                    
Пациент: Иванова Мария Петровна
-------------------------------------------------------
Наименование услуги                 |   Итоговая цена
-------------------------------------------------------
Прием терапевта                     |         3080.00 руб.
Анализ крови (общий)                |         1500.00 руб.
-------------------------------------------------------
ИТОГО К ОПЛАТЕ:                     |         4580.00 руб.


>>> СЦЕНАРИЙ 2: 